In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from chess import Board
import tensorflow as tf
import asyncio
import time
from IPython.display import HTML, display
import ipywidgets as widgets


In [2]:
# Load in the model that you want to use
# and the move encoding

model = tf.keras.models.load_model("Transfer_learning/models/simulated_filtered_model/SSMF_50EPOCHS.keras")
import pickle

with open("Transfer_learning/models/simulated_filtered_model/move_to_int.pkl", "rb") as f:
    move_to_int = pickle.load(f)

with open("Transfer_learning/models/simulated_filtered_model/int_to_move.pkl", "rb") as f:
    int_to_move = pickle.load(f)


In [3]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

In [4]:
import chess
import chess.engine
import chess.pgn
import random
from tqdm import tqdm

# stockfish engine path
# this is the path on my Home machine it should be chnaged if you are using it 
STOCKFISH_PATH = r"C:\Users\p45mi\Downloads\stockfish-windows-x86-64-avx2\stockfish\stockfish-windows-x86-64-avx2.exe"
stockfish_engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)

# engine stats
stockfish_engine.configure({
    "UCI_LimitStrength": True,
    "UCI_Elo": 2000
})

# stockfish model move
def stockfish_move(board):
    result = stockfish_engine.play(board, chess.engine.Limit(time=0.1))
    return result.move.uci()

# our model move
def predict_next_move(board):
    board_matrix = board_to_matrix(board).reshape(1, 8, 8, 12)
    predictions = model.predict(board_matrix, verbose = 0)[0]
    legal_moves = list(board.legal_moves)
    legal_moves_uci = [move.uci() for move in legal_moves]
    sorted_indices = np.argsort(predictions)[::-1]
    for move_index in sorted_indices:
        move = int_to_move[move_index]
        if move in legal_moves_uci:
            return move
    return None

In [5]:
board = chess.Board()

# HTML widget that will contain the SVG
board_html = widgets.HTML(value=chess.svg.board(board=board, size=420))
controls = widgets.HBox([])
step_btn = widgets.Button(description="Step")
autoplay_toggle = widgets.ToggleButton(value=False, description="Autoplay")
speed_slider = widgets.FloatSlider(value=0.8, min=0.05, max=2.0, step=0.05, description="Delay (s)")

controls.children = [step_btn, autoplay_toggle, speed_slider]

def render_board():
    # update the HTML widget value (frontend re-renders)
    svg = chess.svg.board(board=board, size=420)
    board_html.value = svg

def make_move_and_render():
    if board.is_game_over():
        return
    if board.turn == chess.WHITE:
        move_uci = predict_next_move(board)
    else:
        move_uci = stockfish_move(board)
    board.push_uci(move_uci)
    render_board()

step_btn.on_click(lambda b: make_move_and_render())

# Async autoplay (runs in notebook event loop)
_autoplay_task = None

async def autoplay_loop():
    try:
        while autoplay_toggle.value and not board.is_game_over():
            make_move_and_render()
            # non-blocking sleep so notebook can process events and render
            await asyncio.sleep(speed_slider.value)
    finally:
        # ensure toggle resets if game ends
        autoplay_toggle.value = False

def on_autoplay_change(change):
    global _autoplay_task
    if change['new']:
        # start loop (if not already running)
        loop = asyncio.get_event_loop()
        if _autoplay_task is None or _autoplay_task.done():
            _autoplay_task = loop.create_task(autoplay_loop())
    else:
        # stopping; loop will check autoplay_toggle.value and exit
        pass

autoplay_toggle.observe(on_autoplay_change, names='value')

# initial render + layout
render_board()
display(widgets.VBox([controls, board_html]))